# ✈️ AI Travel Planning Agent
**Powered by Claude (Anthropic) + Amadeus API**

This agent searches for real flights, hotels, and activities in real time.
It will only show options that are actually available — if nothing is found it suggests alternatives.

### Setup checklist
1. Get a **free Amadeus sandbox key** at [developers.amadeus.com](https://developers.amadeus.com) → Create App → copy Client ID & Secret
2. Get your **Anthropic API key** at [console.anthropic.com](https://console.anthropic.com)
3. Run all cells top to bottom, then interact in the last cell

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install anthropic amadeus -q

In [ ]:
# ── Cell 2: API Keys & Client Initialisation ──────────────────────────────────
import os

# Option A: paste keys directly (fine for personal Colab sessions)
os.environ.setdefault("ANTHROPIC_API_KEY",  "YOUR_ANTHROPIC_API_KEY")
os.environ.setdefault("AMADEUS_CLIENT_ID",   "YOUR_AMADEUS_CLIENT_ID")
os.environ.setdefault("AMADEUS_CLIENT_SECRET", "YOUR_AMADEUS_CLIENT_SECRET")

# Option B: use Colab Secrets (Secrets panel → add key → enable notebook access)
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"]      = userdata.get("ANTHROPIC_API_KEY")
# os.environ["AMADEUS_CLIENT_ID"]       = userdata.get("AMADEUS_CLIENT_ID")
# os.environ["AMADEUS_CLIENT_SECRET"]   = userdata.get("AMADEUS_CLIENT_SECRET")

import anthropic
from amadeus import Client as AmadeusClient, ResponseError

anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

amadeus = AmadeusClient(
    client_id=os.environ["AMADEUS_CLIENT_ID"],
    client_secret=os.environ["AMADEUS_CLIENT_SECRET"],
    # hostname="production"  # uncomment when you have a production key
)

print("Clients initialised successfully.")

In [ ]:
# ── Cell 3: Tool Functions (real Amadeus API calls) ───────────────────────────
import json
import random
from datetime import datetime, timedelta


# ─── 3.1  Destination lookup ──────────────────────────────────────────────────
def get_destination_info(city_name: str) -> dict:
    """
    Convert a city/airport name to its IATA code and coordinates.
    Returns the best match (usually the main airport or city centre).
    """
    try:
        response = amadeus.reference_data.locations.get(
            keyword=city_name,
            subType="CITY,AIRPORT"
        )
        if not response.data:
            return {"found": False, "message": f"No location found for '{city_name}'"}

        best = response.data[0]
        address = best.get("address", {})
        geo = best.get("geoCode", {})
        return {
            "found": True,
            "iata_code": best.get("iataCode", ""),
            "name": best.get("name", city_name),
            "city": address.get("cityName", ""),
            "country": address.get("countryName", ""),
            "country_code": address.get("countryCode", ""),
            "latitude": geo.get("latitude", None),
            "longitude": geo.get("longitude", None),
        }
    except ResponseError as e:
        return {"found": False, "error": str(e)}


# ─── 3.2  Flight search ───────────────────────────────────────────────────────
def search_flights(
    origin: str,
    destination: str,
    departure_date: str,
    adults: int = 1,
    return_date: str = None,
    travel_class: str = "ECONOMY",
    max_results: int = 5,
) -> dict:
    """
    Search available flight offers.
    origin/destination: IATA codes (e.g. 'JFK', 'CDG').
    departure_date / return_date: 'YYYY-MM-DD'.
    travel_class: ECONOMY | PREMIUM_ECONOMY | BUSINESS | FIRST.
    """
    try:
        params = dict(
            originLocationCode=origin.upper(),
            destinationLocationCode=destination.upper(),
            departureDate=departure_date,
            adults=adults,
            travelClass=travel_class.upper(),
            max=max_results,
            currencyCode="USD",
        )
        if return_date:
            params["returnDate"] = return_date

        response = amadeus.shopping.flight_offers_search.get(**params)
        offers = response.data or []

        if not offers:
            return {"available": False, "message": "No flights found for these criteria."}

        results = []
        for offer in offers:
            itineraries = []
            for itin in offer.get("itineraries", []):
                segs = itin.get("segments", [])
                segments = []
                for s in segs:
                    segments.append({
                        "from": s["departure"]["iataCode"],
                        "to":   s["arrival"]["iataCode"],
                        "departs": s["departure"]["at"],
                        "arrives": s["arrival"]["at"],
                        "airline": s["carrierCode"],
                        "flight": s["carrierCode"] + s["number"],
                        "cabin": (
                            s.get("cabin")
                            or offer.get("travelerPricings", [{}])[0]
                               .get("fareDetailsBySegment", [{}])[0]
                               .get("cabin", travel_class)
                        ),
                    })
                itineraries.append({
                    "duration": itin.get("duration", ""),
                    "stops": len(segs) - 1,
                    "segments": segments,
                })

            price = offer.get("price", {})
            results.append({
                "offer_id": offer.get("id", ""),
                "price_total_usd": price.get("grandTotal", price.get("total", "N/A")),
                "price_per_person": price.get("base", "N/A"),
                "seats_left": offer.get("numberOfBookableSeats", "?"),
                "last_ticketing_date": offer.get("lastTicketingDate", ""),
                "itineraries": itineraries,
            })

        return {"available": True, "count": len(results), "flights": results}

    except ResponseError as e:
        return {"available": False, "error": str(e)}
    except Exception as e:
        return {"available": False, "error": str(e)}


# ─── 3.3  Hotel search ────────────────────────────────────────────────────────
def search_hotels(
    city_code: str,
    check_in: str,
    check_out: str,
    adults: int = 1,
    max_hotels: int = 5,
) -> dict:
    """
    Two-step search: get hotel IDs for the city, then fetch live offers.
    city_code: IATA city code (e.g. 'PAR' for Paris, 'NYC' for New York).
    check_in / check_out: 'YYYY-MM-DD'.
    """
    try:
        # Step 1 — get hotel IDs in city
        hotels_resp = amadeus.reference_data.locations.hotels.by_city.get(
            cityCode=city_code.upper()
        )
        hotels_list = hotels_resp.data or []
        if not hotels_list:
            return {"available": False, "message": f"No hotels found in city code '{city_code}'."}

        # Cap to avoid slow responses; pick the first N
        hotel_ids = [h["hotelId"] for h in hotels_list[:10]]

        # Step 2 — get live availability & prices
        offers_resp = amadeus.shopping.hotel_offers_search.get(
            hotelIds=hotel_ids,
            checkInDate=check_in,
            checkOutDate=check_out,
            adults=adults,
            currency="USD",
            bestRateOnly=True,
        )
        offers_data = offers_resp.data or []
        if not offers_data:
            return {"available": False, "message": "No hotel availability found for those dates."}

        results = []
        for item in offers_data[:max_hotels]:
            hotel = item.get("hotel", {})
            offers = item.get("offers", [])
            if not offers:
                continue
            best_offer = offers[0]
            price = best_offer.get("price", {})
            room = best_offer.get("room", {})
            results.append({
                "offer_id": best_offer.get("id", ""),
                "hotel_id": hotel.get("hotelId", ""),
                "name": hotel.get("name", "Unknown"),
                "rating": hotel.get("rating", "N/A"),
                "latitude": hotel.get("latitude", None),
                "longitude": hotel.get("longitude", None),
                "price_per_night_usd": price.get("total", "N/A"),
                "currency": price.get("currency", "USD"),
                "room_type": room.get("typeEstimated", {}).get("category", "Standard"),
                "board_type": best_offer.get("boardType", "ROOM_ONLY"),
                "cancellation_policy": best_offer.get("policies", {}).get(
                    "cancellations", [{"type": "N/A"}]
                )[0].get("type", "N/A"),
            })

        if not results:
            return {"available": False, "message": "No rooms available for those dates."}

        return {"available": True, "count": len(results), "hotels": results}

    except ResponseError as e:
        return {"available": False, "error": str(e)}
    except Exception as e:
        return {"available": False, "error": str(e)}


# ─── 3.4  Activities / attractions search ─────────────────────────────────────
def search_activities(
    latitude: float,
    longitude: float,
    radius_km: int = 20,
    max_results: int = 8,
) -> dict:
    """
    Search tours, experiences, and attractions near a geographic point.
    Use get_destination_info() first to get lat/lon from a city name.
    """
    try:
        response = amadeus.shopping.activities.get(
            latitude=latitude,
            longitude=longitude,
            radius=radius_km,
        )
        activities = response.data or []
        if not activities:
            return {"available": False, "message": "No activities found near this location."}

        results = []
        for act in activities[:max_results]:
            price = act.get("price", {})
            results.append({
                "id": act.get("id", ""),
                "name": act.get("name", ""),
                "short_description": act.get("shortDescription", ""),
                "rating": act.get("rating", "N/A"),
                "price_usd": price.get("amount", "N/A"),
                "currency": price.get("currencyCode", "USD"),
                "duration_hours": act.get("duration", "N/A"),
                "booking_link": act.get("bookingLink", ""),
                "pictures": act.get("pictures", [])[:1],
            })

        return {"available": True, "count": len(results), "activities": results}

    except ResponseError as e:
        return {"available": False, "error": str(e)}
    except Exception as e:
        return {"available": False, "error": str(e)}


# ─── 3.5  Booking scaffold ────────────────────────────────────────────────────
def book_trip(
    flight_offer_id: str,
    hotel_offer_id: str,
    passenger_first_name: str,
    passenger_last_name: str,
    passenger_email: str,
    passenger_phone: str = "",
) -> dict:
    """
    SCAFFOLD — simulates a booking confirmation.

    TODO (production): Replace the return below with real Amadeus booking calls:
      - Flights : amadeus.booking.flight_orders.post(flight_offer, traveler_info)
      - Hotels  : amadeus.booking.hotel_orders.post(hotel_offer_id, guest_info)
    Both require Amadeus Production credentials (apply via the developer portal).
    The Amadeus sandbox supports booking simulation with test traveller data.
    """
    ref = f"TRV-{random.randint(100000, 999999)}"
    return {
        "status": "SIMULATED_CONFIRMED",
        "booking_reference": ref,
        "passenger": {
            "name": f"{passenger_first_name} {passenger_last_name}",
            "email": passenger_email,
            "phone": passenger_phone,
        },
        "flight": {"offer_id": flight_offer_id, "status": "HELD — awaiting payment"},
        "hotel":  {"offer_id": hotel_offer_id,  "status": "HELD — awaiting payment"},
        "note": (
            "This is a SIMULATED booking (sandbox mode). "
            "No real reservation has been made and no charge has occurred. "
            "Wire up production Amadeus credentials to enable real bookings."
        ),
        "next_step": "Provide payment details to confirm (not yet implemented).",
    }


print("Tool functions loaded.")

In [ ]:
# ── Cell 4: Tool Schemas (Claude-compatible JSON schemas) ─────────────────────

TOOLS = [
    {
        "name": "get_destination_info",
        "description": (
            "Look up a city or airport by name to get its IATA code, country, and GPS coordinates. "
            "Always call this first when the user mentions a destination by name rather than code."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "city_name": {"type": "string", "description": "City or airport name, e.g. 'Paris', 'London Heathrow'"},
            },
            "required": ["city_name"],
        },
    },
    {
        "name": "search_flights",
        "description": (
            "Search for available flights between two airports. "
            "Returns real-time offers with prices and seat availability. "
            "Only use IATA airport codes (e.g. JFK, CDG, LHR). "
            "If no flights are found, try nearby airports or different dates."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "origin":        {"type": "string", "description": "Departure airport IATA code, e.g. 'JFK'"},
                "destination":   {"type": "string", "description": "Arrival airport IATA code, e.g. 'CDG'"},
                "departure_date":{"type": "string", "description": "Departure date in YYYY-MM-DD format"},
                "adults":        {"type": "integer", "description": "Number of adult passengers", "default": 1},
                "return_date":   {"type": "string", "description": "Return date in YYYY-MM-DD (omit for one-way)"},
                "travel_class":  {
                    "type": "string",
                    "enum": ["ECONOMY", "PREMIUM_ECONOMY", "BUSINESS", "FIRST"],
                    "description": "Cabin class",
                    "default": "ECONOMY",
                },
                "max_results":   {"type": "integer", "description": "Max offers to return (1–10)", "default": 5},
            },
            "required": ["origin", "destination", "departure_date"],
        },
    },
    {
        "name": "search_hotels",
        "description": (
            "Search hotels with live availability and pricing for a city. "
            "Uses the city IATA code (not airport code — e.g. 'PAR' for Paris, 'NYC' for New York). "
            "Returns only hotels that have rooms available for the requested dates."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "city_code":  {"type": "string", "description": "City IATA code, e.g. 'PAR', 'NYC', 'LON'"},
                "check_in":   {"type": "string", "description": "Check-in date YYYY-MM-DD"},
                "check_out":  {"type": "string", "description": "Check-out date YYYY-MM-DD"},
                "adults":     {"type": "integer", "description": "Number of guests", "default": 1},
                "max_hotels": {"type": "integer", "description": "Max hotels to return (1–10)", "default": 5},
            },
            "required": ["city_code", "check_in", "check_out"],
        },
    },
    {
        "name": "search_activities",
        "description": (
            "Find tours, experiences, and attractions near a location. "
            "Requires GPS coordinates — use get_destination_info() first to get latitude/longitude. "
            "Returns bookable activities with prices and ratings."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "latitude":    {"type": "number", "description": "Latitude of the destination"},
                "longitude":   {"type": "number", "description": "Longitude of the destination"},
                "radius_km":   {"type": "integer", "description": "Search radius in km", "default": 20},
                "max_results": {"type": "integer", "description": "Max activities to return", "default": 8},
            },
            "required": ["latitude", "longitude"],
        },
    },
    {
        "name": "book_trip",
        "description": (
            "Reserve a flight and hotel for the traveller. "
            "Use the offer_id values returned by search_flights and search_hotels. "
            "Currently runs in sandbox mode — no real charge is made."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "flight_offer_id":      {"type": "string", "description": "Offer ID from search_flights"},
                "hotel_offer_id":       {"type": "string", "description": "Offer ID from search_hotels"},
                "passenger_first_name": {"type": "string"},
                "passenger_last_name":  {"type": "string"},
                "passenger_email":      {"type": "string"},
                "passenger_phone":      {"type": "string", "description": "Optional phone number"},
            },
            "required": [
                "flight_offer_id", "hotel_offer_id",
                "passenger_first_name", "passenger_last_name", "passenger_email",
            ],
        },
    },
]

print(f"{len(TOOLS)} tools registered.")

In [ ]:
# ── Cell 5: Tool Dispatcher ───────────────────────────────────────────────────

def execute_tool(name: str, tool_input: dict) -> str:
    """Route a tool call from Claude to the correct function and return JSON."""
    dispatch = {
        "get_destination_info": get_destination_info,
        "search_flights":        search_flights,
        "search_hotels":         search_hotels,
        "search_activities":     search_activities,
        "book_trip":             book_trip,
    }
    fn = dispatch.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        result = fn(**tool_input)
        return json.dumps(result, ensure_ascii=False)
    except TypeError as e:
        return json.dumps({"error": f"Bad arguments for {name}: {e}"})


print("Tool dispatcher ready.")

In [ ]:
# ── Cell 6: Travel Agent Core ─────────────────────────────────────────────────
from IPython.display import display, Markdown

SYSTEM_PROMPT = """\
You are an expert, friendly travel agent with real-time access to flight, hotel, and activity data.

## How you work
1. When the user describes a trip, call your tools to find real availability.
   - Always call get_destination_info() first to resolve city names to IATA codes.
   - Search flights, hotels, and activities in parallel when possible.
2. ONLY present options that your tools actually returned. Never invent prices, flights, or hotels.
3. If a search returns no results, tell the user clearly and immediately suggest alternatives
   (different dates, nearby airports, different city, different cabin class).
4. When presenting results, use this structure:

   ✈️ **FLIGHTS**  — show top 3 options with price, duration, stops, airline
   🏨 **HOTELS**   — show top 3 options with name, star rating, price/night, cancellation policy
   🎯 **ACTIVITIES** — show top 5 options with name, short description, price, rating
   💡 **RECOMMENDATION** — give a specific pick with reasoning (best value, best experience, etc.)

5. Always show prices in USD with the total trip cost estimate at the end.
6. When the user wants to book, collect their name, email, and phone, then call book_trip().
7. Be concise but warm. Use markdown formatting for readability.
"""

MODEL = "claude-opus-4-6"
MAX_TOKENS = 8192


def run_travel_agent(messages: list, verbose_tools: bool = False) -> tuple[list, str]:
    """
    Run one full agent turn (may involve multiple tool calls).
    Streams Claude's text output in real time.
    Returns (updated_messages, final_text).
    """
    final_text = ""

    while True:
        with anthropic_client.messages.stream(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            thinking={"type": "adaptive"},
            messages=messages,
        ) as stream:
            # ── stream text in real time ──────────────────────────────────────
            streamed_text = ""
            for event in stream:
                if (
                    event.type == "content_block_delta"
                    and hasattr(event.delta, "text")
                ):
                    print(event.delta.text, end="", flush=True)
                    streamed_text += event.delta.text

            response = stream.get_final_message()

        # ── append assistant response to history ──────────────────────────────
        messages.append({"role": "assistant", "content": response.content})

        if streamed_text:
            final_text += streamed_text

        # ── if no tool calls, we're done ──────────────────────────────────────
        if response.stop_reason != "tool_use":
            print()  # newline after streamed output
            break

        # ── handle tool calls ─────────────────────────────────────────────────
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            if verbose_tools:
                print(f"\n[TOOL] {block.name}({json.dumps(block.input, ensure_ascii=False)})")
            else:
                print(f"\n[searching: {block.name}...]")

            result_json = execute_tool(block.name, block.input)

            if verbose_tools:
                # Pretty-print first 300 chars of result
                preview = result_json[:300] + ("..." if len(result_json) > 300 else "")
                print(f"[RESULT] {preview}")

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result_json,
            })

        messages.append({"role": "user", "content": tool_results})
        # loop continues — Claude will now interpret the tool results

    return messages, final_text


print("Travel agent core ready.")

In [ ]:
# ── Cell 7: Initial Trip Request ──────────────────────────────────────────────
# Fill in your trip details below (or leave blanks to let the agent suggest)

print("=" * 60)
print("         AI TRAVEL AGENT — Trip Planner")
print("=" * 60)
print("Press Enter to skip optional fields.\n")

origin          = input("Your departure city or airport (e.g. New York, JFK): ").strip()
destination     = input("Where do you want to go? (or vague, e.g. 'beach in Europe'): ").strip()
departure_date  = input("Departure date (YYYY-MM-DD): ").strip()
return_date     = input("Return date    (YYYY-MM-DD, or Enter for one-way):  ").strip() or None
adults_str      = input("Number of travellers (default 1): ").strip()
adults          = int(adults_str) if adults_str.isdigit() else 1
budget_hint     = input("Budget hint (e.g. budget / mid-range / luxury): ").strip()
extra_prefs     = input("Any preferences? (e.g. direct flights only, sea view, vegan food): ").strip()

# Build the opening user message
user_msg_parts = []
if origin:         user_msg_parts.append(f"I'm flying from {origin}.")
if destination:    user_msg_parts.append(f"I want to go to {destination}.")
if departure_date: user_msg_parts.append(f"Departing on {departure_date}.")
if return_date:    user_msg_parts.append(f"Returning on {return_date}.")
user_msg_parts.append(f"We are {adults} adult(s).")
if budget_hint:    user_msg_parts.append(f"Budget: {budget_hint}.")
if extra_prefs:    user_msg_parts.append(f"Preferences: {extra_prefs}.")
user_msg_parts.append(
    "Please search for available flights, hotels, and top activities, "
    "then give me a full trip plan with your best recommendation."
)

user_message = " ".join(user_msg_parts)

print("\n" + "=" * 60)
print("Agent is planning your trip...\n")

# Initialise message history and run the agent
conversation = [{"role": "user", "content": user_message}]
conversation, _ = run_travel_agent(conversation, verbose_tools=False)

In [ ]:
# ── Cell 8: Follow-up Chat Loop ───────────────────────────────────────────────
# Keep refining the plan or trigger a booking.
# Type  'book'  to start the booking flow.
# Type  'quit'  or  'exit'  to end.
# Type  'debug' to toggle verbose tool output.

verbose = False

print("\n" + "=" * 60)
print("Follow-up chat — refine your plan or ask for alternatives.")
print("Commands: 'book', 'debug', 'quit'")
print("=" * 60)

while True:
    try:
        user_input = input("\nYou: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nSession ended.")
        break

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "bye"):
        print("Safe travels! 🌍")
        break

    if user_input.lower() == "debug":
        verbose = not verbose
        print(f"[debug mode {'ON' if verbose else 'OFF'}]")
        continue

    if user_input.lower() == "book":
        print("\nLet's collect your details for the booking.")
        first = input("  First name: ").strip()
        last  = input("  Last name:  ").strip()
        email = input("  Email:      ").strip()
        phone = input("  Phone (optional): ").strip()
        user_input = (
            f"Please book the trip for {first} {last}, email {email}"
            + (f", phone {phone}" if phone else "")
            + ". Use the best flight and hotel options you found earlier."
        )

    print()
    conversation.append({"role": "user", "content": user_input})
    conversation, _ = run_travel_agent(conversation, verbose_tools=verbose)